# ERα docking + QSAR pipeline (Google Colab)
Run the cells **top to bottom**. Read the short explanation above each step so you can explain it in an interview.
Runtime: about 30-60 minutes (docking is the slow part).

## 0. Get the code
Upload `ERalpha-docking-qsar.zip` when asked (or clone your GitHub repo instead).

In [ ]:
import os
if not os.path.exists("scripts/common.py"):
    from google.colab import files
    files.upload()                       # choose ERalpha-docking-qsar.zip
    !unzip -q -o ERalpha-docking-qsar.zip
    %cd ERalpha-docking-qsar
os.environ["PATH"] = os.getcwd() + "/bin:" + os.environ["PATH"]

## 0b. Install packages, AutoDock Vina and download the 3ERT structure

In [ ]:
!bash scripts/00_setup.sh

## 1. Prepare the protein
Keep chain A, remove water, add hydrogens, define the docking box around the co-crystal ligand 4-hydroxytamoxifen (OHT).

In [ ]:
!python scripts/01_prepare_protein.py

## 2. Validate docking by re-docking OHT
RMSD < 2 Å between docked and crystal pose means the set-up is trustworthy.

In [ ]:
!python scripts/02_redock.py

In [ ]:
from IPython.display import Image, display
display(Image('figures/redocking_overlay.png', width=420))

## 3. ChEMBL data for ERα (CHEMBL206)
IC50 in nM, exact values; pIC50 = 9 − log10(IC50 nM). Active = pIC50 ≥ 6.

If the API is slow/fails, download the CSV from the ChEMBL website, save it as `data/raw/chembl206_activities.csv` and use `--source csv`.

In [ ]:
!python scripts/03_chembl_data.py --source api

In [ ]:
display(Image('figures/pIC50_distribution.png', width=450))

## 4. QSAR model
Random Forest on Morgan fingerprints, scaffold split, scaffold-grouped CV, Y-randomisation, applicability domain.

In [ ]:
!python scripts/04_qsar.py

In [ ]:
display(Image('figures/qsar_scatter_AD.png', width=800)); display(Image('figures/qsar_top_bits.png', width=500))

## 5. Virtual screening
Docks hold-out compounds and compares Vina scores with experiment (use `--workers` = number of CPU cores).

In [ ]:
import os
!python scripts/05_screen_dock.py --n 100 --workers {os.cpu_count()}

In [ ]:
display(Image('figures/docking_vs_pIC50.png', width=800))

## 6. Interaction analysis (ProLIF)
Which residues (Glu353, Arg394, His524, Leu387 …) do the top hits contact, compared with OHT?

In [ ]:
!python scripts/06_interactions.py

In [ ]:
display(Image('figures/interaction_heatmap.png', width=700))

In [ ]:
from IPython.display import HTML
HTML(open('results/top1_complex.html').read())   # interactive 3D view

## 7. Summary table + download everything

In [ ]:
!python scripts/07_make_report.py

In [ ]:
!zip -q -r results_bundle.zip figures results data/processed/chembl_clean.csv data/processed/train.csv data/processed/test.csv -x "results/poses_all/*"
from google.colab import files
files.download("results_bundle.zip")